In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os
from pandas.errors import EmptyDataError

# TEST phenotypes

In [2]:
dmeta = pd.read_csv("../DATA_intermediate/23_filter_EPN_indiv_metrics.tsv", sep="\t", header=0)
dmeta["POP_SITE"] = dmeta.apply(lambda x: str(x["POP"]) + "_" + x["SITE_ID"], axis=1)
dmeta = dmeta[["POP_SITE", "group"]].drop_duplicates().rename(columns={"group":"group_cluster"}).reset_index(drop=True)
print(dmeta.shape)
dmeta.head()

#dmeta[dmeta["group_cluster"] == "1530_AC"]
dmeta["group_cluster"].drop_duplicates()

(128, 2)


0     Central
3        East
4        West
27         ME
30         WI
Name: group_cluster, dtype: object

In [3]:
dpheno_ = pd.read_csv("../44_train_and_test/00_combine_samples_means.tsv", sep="\t", header=0)
dpheno = pd.merge(dpheno_, dmeta, on = "POP_SITE", how = "left")
dpheno = dpheno[dpheno['SET'] == 'TEST'].reset_index(drop=True)
###dpheno = dpheno[~dpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
print(dpheno.shape)
dpheno["sample"] = dpheno["POP_SITE"]
dpheno.head()

(2048, 9)


,Trait_name,POP_SITE,POP_ID,SITE_ID,SET,mean,sd,n,group_cluster,sample
0,Average_Ring_Density,1329_CH,1329,CH,TEST,476.575417,45.597986,8,East,1329_CH
1,Average_Ring_Density,1329_ML,1329,ML,TEST,480.010872,108.572664,3,East,1329_ML
2,Average_Ring_Density,1528_ML,1528,ML,TEST,514.502734,62.179284,3,East,1528_ML
3,Average_Ring_Density,1530_AC,1530,AC,TEST,553.321396,48.783433,14,NaN,1530_AC
4,Average_Ring_Density,1530_CH,1530,CH,TEST,526.616667,93.195567,6,East,1530_CH


In [4]:
dpheno[dpheno["n"] < 3].shape

(48, 10)

In [5]:
dpheno = dpheno[dpheno['n']>2].reset_index(drop=True)
print(dpheno.shape)

(2000, 10)


# TRAIN phenotypes

In [6]:
tpheno_ = pd.read_csv("../44_train_and_test/00_combine_samples.tsv", sep="\t", header=0)
tpheno = pd.merge(tpheno_, dmeta, on = "POP_SITE", how = "left")
tpheno = tpheno[tpheno['SET'] == 'TRAIN'].reset_index(drop=True)
###tpheno = tpheno[~tpheno['group_cluster'].isin(['ME','WI'])].reset_index(drop=True)
tpheno["sample"] = tpheno["POP_SITE"]
print(tpheno.shape)
tpheno.head()

#tpheno[tpheno.isna().any(axis=1)]
tpheno_grouped = tpheno.dropna().groupby(['sample','Trait_name']).agg(n = ('genotype','count')).reset_index()
print(tpheno_grouped[tpheno_grouped['n'] < 3].shape)
large_samples = tpheno_grouped[tpheno_grouped['n'] > 2].reset_index(drop=True)

tpheno_flt = pd.merge(large_samples, tpheno, on = ['sample','Trait_name'], how = 'left')
print(tpheno_flt.shape)
tpheno_flt.head()

(18282, 11)
(24, 3)
(17169, 12)


,sample,Trait_name,n,genotype,POP_ID,SITE_ID,call_rate,good_genotype,value,SET,POP_SITE,group_cluster
0,1329_CH,Average_Ring_Density,7,E353-B3_4_11_1329_161,1329,CH,0.909280,True,608.600000,TRAIN,1329_CH,East
1,1329_CH,Average_Ring_Density,7,E353-B3_4_15_1329_162,1329,CH,0.861028,True,470.500000,TRAIN,1329_CH,East
2,1329_CH,Average_Ring_Density,7,E353-B3_4_6_1329_158,1329,CH,0.908800,True,459.300000,TRAIN,1329_CH,East
3,1329_CH,Average_Ring_Density,7,E353-B3_5_12_1329_616,1329,CH,0.898735,True,440.196875,TRAIN,1329_CH,East
4,1329_CH,Average_Ring_Density,7,E353-B3_5_13_1329_617,1329,CH,0.907139,True,424.300000,TRAIN,1329_CH,East


# Combining genetic offset sets

In [9]:
traits = ["Height", "Biomass_Increment", "Biomass_Increment_1980", "Biomass_Increment_1985", 
                "Biomass_Increment_1990", "Biomass_Increment_1995", "Biomass_Increment_2000", 
                "Biomass_Increment_2005", "Biomass_Increment_2010", "Biomass_Increment_2015",
                "Average_Ring_Density", "DBH", "Rs", "Rl", "Rr","Rc"]

gardens = ["PR","ML","CH","AC"]
#groups = ["East","Central","West"]
upper_limits = ['2','4','6','8','10','12','14','100']
iterations = ["1", "2", "3", "4", "5"]

#dsets = ["100","1000","lfmm","RDA","RDAcorrected", "10000","all"]
#dsets = ["100","1000","10000","lfmm","RDA","RDAcorrected"]
#dsets = ["1000"]
#dsets

In [10]:
D = []
M = []
for trait in traits:
    for garden in gardens:
        for upper_limit in upper_limits:
            for iteration in iterations:
                
                file_path = "../993_run_rda_exclude_geno/results/01_run_gradient_forest_" + garden + "_" + upper_limit + "_" + iteration + "_" + trait + ".tsv"
                if os.path.exists(file_path):
                    try:
                        df_ = pd.read_csv(file_path, sep="\t", header=0)
                        df_["garden"] = garden
                        df_["marker"] = "1000"
                        df_["trait"] = trait
                        df_["upper_limit"] = upper_limit
                        df_["iteration"] = iteration
                        D.append(df_)
                    except EmptyDataError:
                        print("File is empty")
                        
    
                meta_path = "../953_garden_offset_run_exclude_geno/genotypes_trees/01_samples_POP_" + trait + "_" + upper_limit + "_" + iteration + ".tsv"
                if os.path.exists(meta_path):
                    try:
                        dm_ = pd.read_csv(meta_path, sep="\t", header=0)
                        dm_["garden"] = garden
                        dm_["marker"] = "1000"
                        dm_["trait"] = trait
                        dm_["upper_limit"] = upper_limit
                        dm_["iteration"] = iteration
                        M.append(dm_)
                    except EmptyDataError:
                        print("File is empty")

dD = pd.concat(D)
dM = pd.concat(M)
dM = dM.drop(columns = ["Unnamed: 0"]).rename(columns = {"POP_SITE":"sample"}).reset_index(drop=True)
print(dD.shape)
dD.head()

(80760, 19)


,Trait_name,sample,SITE_ID,group,PC1,PC2,PC3,now_at1,now_at2,now_at3,new_at1,new_at2,new_at3,offset_rda,garden,marker,trait,upper_limit,iteration
0,Height,2209_PR,PR,West,12.351204,-1.598833,0.851360,-0.076611,0.078353,-0.085735,-0.09554,0.00354,0.04641,0.153028,PR,1000,Height,2,1
1,Height,325_PR,PR,Central,2.506557,-3.166880,1.958590,0.152059,0.066594,-0.007166,-0.09554,0.00354,0.04641,0.261058,PR,1000,Height,2,1
2,Height,4351_PR,PR,Central,-6.244846,1.531312,-1.399408,0.010306,-0.038117,0.097094,-0.09554,0.00354,0.04641,0.124530,PR,1000,Height,2,1
3,Height,4420_PR,PR,West,6.170760,3.960606,-2.183338,-0.231593,-0.030799,-0.020754,-0.09554,0.00354,0.04641,0.155566,PR,1000,Height,2,1
4,Height,6805_PR,PR,East,0.111999,-2.575310,0.678848,0.086586,0.049158,0.036955,-0.09554,0.00354,0.04641,0.187990,PR,1000,Height,2,1


In [11]:
print(len(M), len(D))

2560 2520


In [12]:
dM.head()
#dD.head()

,sample,genotype,garden,marker,trait,upper_limit,iteration
0,1329_CH,E353-B3_4_11_1329_161,PR,1000,Height,2,1
1,1329_CH,E353-B3_5_12_1329_616,PR,1000,Height,2,1
2,1329_ML,E353-B1_4_9_1329_303,PR,1000,Height,2,1
3,1329_ML,E353-B1_6_12_1329_707,PR,1000,Height,2,1
4,1528_ML,E353-B1_3_11_1528_010,PR,1000,Height,2,1


In [13]:
dD_sub = dD[["sample","SITE_ID","garden","offset_rda","marker","trait","upper_limit","iteration","group"]].reset_index(drop=True)
dM_sub = pd.merge(dD_sub[["trait","sample","garden","upper_limit","iteration"]], dM, on = ["trait","sample","garden","upper_limit","iteration"], how = "left")
dM_sub["SITE_ID"] = dM_sub["garden"]
print(dD_sub.shape, dM_sub.shape)
dD_sub.head()

(80760, 9) (509450, 8)


,sample,SITE_ID,garden,offset_rda,marker,trait,upper_limit,iteration,group
0,2209_PR,PR,PR,0.153028,1000,Height,2,1,West
1,325_PR,PR,PR,0.261058,1000,Height,2,1,Central
2,4351_PR,PR,PR,0.124530,1000,Height,2,1,Central
3,4420_PR,PR,PR,0.155566,1000,Height,2,1,West
4,6805_PR,PR,PR,0.187990,1000,Height,2,1,East


In [14]:
dM_sub.head()

,trait,sample,garden,upper_limit,iteration,genotype,marker,SITE_ID
0,Height,2209_PR,PR,2,1,441-G348-1-17-1,1000,PR
1,Height,2209_PR,PR,2,1,775-G348-3-17-7,1000,PR
2,Height,325_PR,PR,2,1,734-G348-2-23-2,1000,PR
3,Height,325_PR,PR,2,1,923-G348-3-23-3,1000,PR
4,Height,4351_PR,PR,2,1,828-G348-3-24-2,1000,PR


### Test

In [15]:
dD_merge = pd.merge(dD_sub, dpheno, left_on = ["sample","SITE_ID","trait"], right_on = ["sample","SITE_ID","Trait_name"], how = "left")
print(dD_merge.shape)
dD_merge.head()

(80760, 17)


,sample,SITE_ID,garden,offset_rda,marker,trait,upper_limit,iteration,group,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group_cluster
0,2209_PR,PR,PR,0.153028,1000,Height,2,1,West,Height,2209_PR,2209.0,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,PR,0.261058,1000,Height,2,1,Central,Height,325_PR,325.0,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,PR,0.124530,1000,Height,2,1,Central,Height,4351_PR,4351.0,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,PR,0.155566,1000,Height,2,1,West,Height,4420_PR,4420.0,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,PR,0.187990,1000,Height,2,1,East,Height,6805_PR,6805.0,TEST,1353.974749,229.855027,7.0,East


### Train

In [16]:
dM_sub.head()
tD_merge = pd.merge(dM_sub, tpheno, left_on = ["sample","SITE_ID","trait","genotype"], right_on = ["sample","SITE_ID","Trait_name","genotype"], how = "left")
print(dM_sub.shape, tpheno.shape, tD_merge.shape)
tD_merge.head()

# Merging genotypes with TRAIN phenotypes and cal;culating mean
tG_merge = tD_merge.groupby(["trait","sample","SITE_ID","garden","upper_limit","iteration","SET"]).agg(mean = ('value','mean'),
                                                                                                      sd = ('value','std'),
                                                                                                      n = ('value','count')).reset_index()
tG_merge.head()


(509450, 8) (18282, 11) (509450, 16)


,trait,sample,SITE_ID,garden,upper_limit,iteration,SET,mean,sd,n
0,Average_Ring_Density,1329_CH,CH,CH,10,1,TRAIN,465.670982,65.589447,7
1,Average_Ring_Density,1329_CH,CH,CH,10,2,TRAIN,465.670982,65.589447,7
2,Average_Ring_Density,1329_CH,CH,CH,10,3,TRAIN,465.670982,65.589447,7
3,Average_Ring_Density,1329_CH,CH,CH,10,4,TRAIN,465.670982,65.589447,7
4,Average_Ring_Density,1329_CH,CH,CH,10,5,TRAIN,465.670982,65.589447,7


In [17]:
tR_merge = pd.merge(dD_sub, tG_merge, left_on = ["trait", "sample","SITE_ID","garden","upper_limit","iteration"],
                    right_on = ["trait", "sample","SITE_ID","garden","upper_limit","iteration"], how = "left")
print(tR_merge.shape, dD_sub.shape, tG_merge.shape)
tR_merge.head()

# Removing samples that were filtered from TRAIN (low sample size) or are not there (no phenotypes measured)
tR_merge = tR_merge.dropna().reset_index(drop=True)

print(tR_merge.shape)
tR_merge.head()

(80760, 13) (80760, 9) (66800, 10)
(64442, 13)


,sample,SITE_ID,garden,offset_rda,marker,trait,upper_limit,iteration,group,SET,mean,sd,n
0,2209_PR,PR,PR,0.153028,1000,Height,2,1,West,TRAIN,840.525258,417.193001,2.0
1,325_PR,PR,PR,0.261058,1000,Height,2,1,Central,TRAIN,1027.391145,162.634560,2.0
2,4351_PR,PR,PR,0.124530,1000,Height,2,1,Central,TRAIN,1363.598605,162.634560,2.0
3,4420_PR,PR,PR,0.155566,1000,Height,2,1,West,TRAIN,668.153897,113.137085,2.0
4,6805_PR,PR,PR,0.187990,1000,Height,2,1,East,TRAIN,1303.974749,14.142136,2.0


### Calculating rho

In [18]:
def spearman_group(df, col_a, col_b, group_cols):
    results = []
    for name, g in df.groupby(group_cols):
        if len(g) < 3:
            rho, p = np.nan, np.nan
        elif ((len(g[col_a].dropna())<3) or (len(g[col_b].dropna())<3)):
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(g[col_a], g[col_b], nan_policy='omit')
        # name is tuple if multiple group columns
        name = name if isinstance(name, tuple) else (name,)
        results.append((*name, rho, p, len(g)))
            
    res_df = pd.DataFrame(results, columns=list(group_cols) + ['spearman_rho', 'p_value', 'n_samps'])
    
    # Adjust p-values
    mask = res_df['p_value'].notna()
    res_df['p_adj'] = np.nan
    if mask.any():
        res_df.loc[mask, 'p_adj'] = multipletests(res_df.loc[mask, 'p_value'], method='fdr_bh')[1]

    return res_df

### Test

In [19]:
df_cors = spearman_group(dD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','upper_limit','iteration'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,marker,upper_limit,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.776210,2.874485e-07,32,0.000036,minus,TEST
1,AC,Average_Ring_Density,1000,10,2,-0.766935,4.852932e-07,32,0.000036,minus,TEST
2,AC,Average_Ring_Density,1000,10,3,-0.770565,3.965082e-07,32,0.000036,minus,TEST
3,AC,Average_Ring_Density,1000,10,4,-0.770968,3.876184e-07,32,0.000036,minus,TEST
4,AC,Average_Ring_Density,1000,10,5,-0.775000,3.081971e-07,32,0.000036,minus,TEST


### Train

In [20]:
dt_cors = spearman_group(tR_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','upper_limit','iteration'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,marker,upper_limit,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.800000,0.009628,9,0.048333,minus,TRAIN
1,AC,Average_Ring_Density,1000,10,2,-0.683333,0.042442,9,0.155059,minus,TRAIN
2,AC,Average_Ring_Density,1000,10,3,-0.716667,0.029818,9,0.119655,minus,TRAIN
3,AC,Average_Ring_Density,1000,10,4,-0.716667,0.029818,9,0.119655,minus,TRAIN
4,AC,Average_Ring_Density,1000,10,5,-0.783333,0.012520,9,0.058395,minus,TRAIN


### Combined

In [21]:
dcombined = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined.head()

,SITE_ID,trait,marker,upper_limit,iteration,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,-0.776210,2.874485e-07,32,0.000036,minus,TEST
1,AC,Average_Ring_Density,1000,10,2,-0.766935,4.852932e-07,32,0.000036,minus,TEST
2,AC,Average_Ring_Density,1000,10,3,-0.770565,3.965082e-07,32,0.000036,minus,TEST
3,AC,Average_Ring_Density,1000,10,4,-0.770968,3.876184e-07,32,0.000036,minus,TEST
4,AC,Average_Ring_Density,1000,10,5,-0.775000,3.081971e-07,32,0.000036,minus,TEST


In [22]:
dcombined.to_csv("06_garden_geno_rda.tsv", sep="\t", header=True, index=False)
dcombined.to_excel("06_garden_geno_rda.xlsx")

# With cluster

In [23]:
df_cors = spearman_group(dD_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','upper_limit','iteration','group'])
df_cors['sign'] = df_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
df_cors["SET"] = "TEST"
df_cors.head()

,SITE_ID,trait,marker,upper_limit,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,-0.615385,0.033170,12,0.342995,minus,TEST
1,AC,Average_Ring_Density,1000,10,1,East,-0.430303,0.214492,10,0.710172,minus,TEST
2,AC,Average_Ring_Density,1000,10,1,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,1000,10,1,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,1000,10,1,West,-0.200000,0.747060,6,0.876613,minus,TEST


In [24]:
dt_cors = spearman_group(tR_merge, 'offset_rda', 'mean', ['SITE_ID','trait','marker','upper_limit','iteration', 'group'])
dt_cors['sign'] = dt_cors['spearman_rho'].apply(lambda x: 'plus' if x >0 else 'minus')
dt_cors["SET"] = "TRAIN"
dt_cors.head()

,SITE_ID,trait,marker,upper_limit,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,-0.3,0.623838,5,0.876625,minus,TRAIN
1,AC,Average_Ring_Density,1000,10,1,East,NaN,NaN,2,NaN,minus,TRAIN
2,AC,Average_Ring_Density,1000,10,1,West,NaN,NaN,2,NaN,minus,TRAIN
3,AC,Average_Ring_Density,1000,10,2,Central,0.1,0.872889,5,0.951709,plus,TRAIN
4,AC,Average_Ring_Density,1000,10,2,East,NaN,NaN,2,NaN,minus,TRAIN


In [25]:
dcombined2 = pd.concat([df_cors, dt_cors], ignore_index=True)
dcombined2.head()

,SITE_ID,trait,marker,upper_limit,iteration,group,spearman_rho,p_value,n_samps,p_adj,sign,SET
0,AC,Average_Ring_Density,1000,10,1,Central,-0.615385,0.033170,12,0.342995,minus,TEST
1,AC,Average_Ring_Density,1000,10,1,East,-0.430303,0.214492,10,0.710172,minus,TEST
2,AC,Average_Ring_Density,1000,10,1,ME,NaN,NaN,1,NaN,minus,TEST
3,AC,Average_Ring_Density,1000,10,1,WI,NaN,NaN,1,NaN,minus,TEST
4,AC,Average_Ring_Density,1000,10,1,West,-0.200000,0.747060,6,0.876613,minus,TEST


In [26]:
dcombined2.to_csv("06_garden_geno_rda_clusters.tsv", sep="\t", header=True, index=False)

# END